## 02 — Querying and Benchmarking

The index is built. Now we measure whether it is actually faster.

This notebook:
1. Verifies the grid query returns the same results as the linear scan
2. Benchmarks both methods across viewports at different zoom levels
3. Shows where the grid wins — and where it doesn't
4. Discusses what comes after a uniform grid (R-trees, quadtrees)

## Setup — Bring Both Methods Together

In [2]:
import json
import math
import time
from pathlib import Path


def find_data_file(*parts):
    """Find lesson data from common notebook working directories."""
    filename = Path(*parts).name
    candidates = [
        Path("../../data").joinpath(*parts),
        Path("../../../data").joinpath(*parts),
        Path("data").joinpath(*parts),
        Path("Assignments_Completed/03-Data_Manager/data").joinpath(*parts),
    ]

    for path in candidates:
        if path.exists():
            return path

    matches = [path for path in Path.cwd().rglob(filename) if path.parts[-len(parts):] == parts]
    if matches:
        return matches[0]

    raise FileNotFoundError(f"Could not find {'/'.join(parts)} from {Path.cwd()}")


with open(find_data_file("lod", "railroads_fine.geojson")) as f:
    fine = json.load(f)

features = fine["features"]
print(f"Fine LOD: {len(features):,} features")

Fine LOD: 25,413 features


In [4]:
# ── helper functions ──────────────────────────────────────────────────────────

def feature_bbox(feature):
    coords = feature["geometry"]["coordinates"]
    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]
    return [min(lons), min(lats), max(lons), max(lats)]


def bbox_intersects(bbox_a, bbox_b):
    a0, a1, a2, a3 = bbox_a
    b0, b1, b2, b3 = bbox_b
    if a2 < b0: return False
    if a0 > b2: return False
    if a3 < b1: return False
    if a1 > b3: return False
    return True


def linear_cull(features, viewport_bbox):
    """Original linear scan from Module 03."""
    return [f for f in features if bbox_intersects(feature_bbox(f), viewport_bbox)]


# ── grid index ────────────────────────────────────────────────────────────────

class GridIndex:
    def __init__(self, cell_size=10.0):
        self.cell_size = cell_size
        self.cells = {}
        self.n_features = 0

    def _cells_for_bbox(self, bbox):
        lon_min, lat_min, lon_max, lat_max = bbox
        n_cols = int(360 / self.cell_size)
        n_rows = int(180 / self.cell_size)

        col_min = max(0, min(n_cols - 1, math.floor((lon_min + 180) / self.cell_size)))
        col_max = max(0, min(n_cols - 1, math.floor((lon_max + 180) / self.cell_size)))
        row_min = max(0, min(n_rows - 1, math.floor((lat_min +  90) / self.cell_size)))
        row_max = max(0, min(n_rows - 1, math.floor((lat_max +  90) / self.cell_size)))
        return [
            (col, row)
            for col in range(col_min, col_max + 1)
            for row in range(row_min, row_max + 1)
        ]

    def build(self, features):
        self.cells = {}
        self.n_features = len(features)
        for idx, feature in enumerate(features):
            for cell in self._cells_for_bbox(feature_bbox(feature)):
                if cell not in self.cells:
                    self.cells[cell] = []
                self.cells[cell].append((idx, feature))

    def query(self, viewport_bbox):
        seen = set()
        results = []
        for cell in self._cells_for_bbox(viewport_bbox):
            for idx, feature in self.cells.get(cell, []):
                if idx not in seen:
                    seen.add(idx)
                    if bbox_intersects(feature_bbox(feature), viewport_bbox):
                        results.append(feature)
        return results

In [5]:
# Build the index
index = GridIndex(cell_size=10.0)
index.build(features)
print("Index built.")

Index built.


## Step 1 — Verify Correctness

Before benchmarking, confirm the grid query returns the same features as the linear scan. The sets of feature indices must match.

In [6]:
def feature_index(feature, features):
    """Return the index of a feature in the list (by identity)."""
    return next(i for i, f in enumerate(features) if f is feature)


test_viewports = [
    ("Europe",      [-10, 35,  40, 70]),
    ("France",      [ -5, 42,  10, 52]),
    ("Paris",       [  1, 48,   4, 50]),
]

all_match = True
for name, vp in test_viewports:
    linear = set(id(f) for f in linear_cull(features, vp))
    grid   = set(id(f) for f in index.query(vp))
    match  = linear == grid
    if not match:
        all_match = False
        only_linear = linear - grid
        only_grid   = grid   - linear
        print(f"  MISMATCH {name}: only in linear={len(only_linear)}, only in grid={len(only_grid)}")
    else:
        print(f"  MATCH  {name}:  {len(linear)} features")

print()
print("All results match" if all_match else "MISMATCHES FOUND — check implementation")

  MATCH  Europe:  12626 features
  MATCH  France:  1598 features
  MATCH  Paris:  51 features

All results match


## Step 2 — Benchmark

Now time both methods across five viewports representing zoom levels 2 through 13.

In [7]:
RUNS = 100  # repeat each query this many times to get stable timings

viewports = [
    ("World z2",         [-180, -90,  180,  90]),
    ("Europe z5",        [  -10,  35,   40,  70]),
    ("France z7",        [   -5,  42,   10,  52]),
    ("Paris z10",        [    1,  48,    4,  50]),
    ("Central Paris z13",[  2.3, 48.8, 2.4, 48.9]),
]

print(f"{'Viewport':<22} {'Linear (ms)':>13} {'Grid (ms)':>11} {'Speedup':>9} {'Features':>10}")
print("-" * 70)

for name, vp in viewports:
    # Time linear scan
    t0 = time.perf_counter()
    for _ in range(RUNS):
        linear_result = linear_cull(features, vp)
    linear_ms = (time.perf_counter() - t0) / RUNS * 1000

    # Time grid query
    t0 = time.perf_counter()
    for _ in range(RUNS):
        grid_result = index.query(vp)
    grid_ms = (time.perf_counter() - t0) / RUNS * 1000

    speedup = linear_ms / grid_ms if grid_ms > 0 else float('inf')
    n = len(linear_result)

    print(f"{name:<22} {linear_ms:>12.3f} {grid_ms:>10.3f} {speedup:>8.1f}x {n:>10,}")

Viewport                 Linear (ms)   Grid (ms)   Speedup   Features
----------------------------------------------------------------------
World z2                     23.137     27.832      0.8x     25,413
Europe z5                    22.161     15.023      1.5x     12,626
France z7                    21.810      6.202      3.5x      1,598
Paris z10                    20.963      1.443     14.5x         51
Central Paris z13            20.630      0.799     25.8x         11


## Reading the Results

The speedup should increase as the viewport gets smaller:

- **World zoom** — the viewport covers all cells. The grid visits every cell and deduplicates, which adds overhead vs. a plain linear scan. The grid is **slower** here.
- **City zoom** — the viewport touches 1–2 cells with a small number of features. The grid is **much faster** than scanning 20,000 features.

This is the fundamental characteristic of spatial indexes: they help most when queries are small relative to the total dataset.

## Live Map with Grid Index

Replace the linear scan in the Module 03 live map with the grid query.

In [8]:
from ipyleaflet import Map, GeoJSON
import ipywidgets as widgets

def leaflet_bounds_to_bbox(bounds):
    (lat_min, lon_min), (lat_max, lon_max) = bounds
    return [lon_min, lat_min, lon_max, lat_max]

m = Map(center=[48.5, 2.5], zoom=6)
status = widgets.Label(value="Pan or zoom to trigger index query")

layer = GeoJSON(
    data={"type": "FeatureCollection", "features": []},
    style={"color": "#cc3300", "weight": 1.5, "opacity": 0.8}
)
m.add(layer)

def update_layer(*args):
    if not m.bounds:
        return
    vp = leaflet_bounds_to_bbox(m.bounds)

    t0 = time.perf_counter()
    visible = index.query(vp)
    elapsed_ms = (time.perf_counter() - t0) * 1000

    layer.data = {"type": "FeatureCollection", "features": visible}
    status.value = (
        f"{len(visible):,} features  |  "
        f"query: {elapsed_ms:.2f}ms  |  "
        f"viewport: {[round(v, 2) for v in vp]}"
    )

m.observe(update_layer, names=["bounds"])
update_layer()

widgets.VBox([m, status])

## Beyond the Uniform Grid

The grid index has one known weakness: **uneven data distribution**.

Railroads are not uniformly distributed across the world. Dense cells (Western Europe, eastern US) have hundreds of features. Empty cells (oceans, poles) have none.

A uniform grid cannot adapt to this — it gives every cell the same size regardless of how many features it holds.

More advanced spatial indexes solve this:

| Index | Key idea |
|-------|----------|
| **Quadtree** | Recursively subdivide cells that are too full |
| **R-tree** | Pack features into tight bounding rectangles that adapt to density |
| **KD-tree** | Binary spatial partitioning on alternating axes |

Python's `shapely` STRtree is a production R-tree. Now that you have built a grid index from scratch, you understand *why* it exists and what it improves over.

We will not build a full R-tree here — that is the industrial tool. But you know the problem it solves.

## Exercise A

The benchmark shows the grid is slower than linear scan at world zoom. Explain **why** — trace through what the grid query does when `viewport_bbox` covers all 648 cells, and compare the work to a plain list comprehension.

In [9]:
# Exercise A answer:
# At world zoom the viewport touches the whole 10-degree grid:
# 36 longitude columns * 18 latitude rows = 648 cells.
#
# A linear scan does one simple pass over the feature list:
#   compute each feature bbox, test intersection, append matches.
#
# The grid query has extra overhead at this scale:
#   1. build the list of all 648 touched cells,
#   2. look up each cell in the dictionary,
#   3. walk every feature reference stored in those cells,
#   4. check the seen set because the same feature can be stored in many cells,
#   5. run the final bbox intersection check before returning a feature.
#
# Because world zoom includes almost everything anyway, the grid cannot skip much work.
# It ends up doing the same broad scan plus dictionary lookups and deduplication.

def query_stats(grid, viewport_bbox):
    cells = grid._cells_for_bbox(viewport_bbox)
    reference_visits = 0
    duplicate_refs = 0
    seen = set()
    results = []

    for cell in cells:
        for idx, feature in grid.cells.get(cell, []):
            reference_visits += 1
            if idx in seen:
                duplicate_refs += 1
                continue
            seen.add(idx)
            if bbox_intersects(feature_bbox(feature), viewport_bbox):
                results.append(feature)

    return {
        "cells_touched": len(cells),
        "reference_visits": reference_visits,
        "duplicate_refs": duplicate_refs,
        "unique_candidates": len(seen),
        "results": len(results),
    }


world_stats = query_stats(index, [-180, -90, 180, 90])
print("World viewport grid work:")
for key, value in world_stats.items():
    print(f"  {key}: {value:,}")

print(f"\nLinear scan feature checks: {len(features):,}")

World viewport grid work:
  cells_touched: 648
  reference_visits: 26,571
  duplicate_refs: 1,158
  unique_candidates: 25,413
  results: 25,413

Linear scan feature checks: 25,413


## Exercise B

Try Shapely's `STRtree` as an alternative index. Build it from the fine LOD features and time it against the same five viewports.

```python
from shapely.strtree import STRtree
from shapely.geometry import box
```

Compare the STRtree query times to your grid index. Where does the R-tree win?

In [10]:
# Build a Shapely STRtree from the fine LOD feature bounding boxes.
# The tree returns candidate geometries/indices quickly; the final bbox check
# keeps the result definition identical to linear_cull() and GridIndex.query().

try:
    from numbers import Integral
    from shapely.geometry import box
    from shapely.strtree import STRtree

    feature_bboxes = [feature_bbox(feature) for feature in features]
    bbox_geoms = [box(*bbox) for bbox in feature_bboxes]

    t0 = time.perf_counter()
    tree = STRtree(bbox_geoms)
    build_ms = (time.perf_counter() - t0) * 1000

    # Shapely 2 returns integer indices. Shapely 1.x returns geometry objects.
    geom_id_to_idx = {id(geom): idx for idx, geom in enumerate(bbox_geoms)}

    def strtree_cull(viewport_bbox):
        query_geom = box(*viewport_bbox)
        raw_matches = tree.query(query_geom)
        result = []

        for item in raw_matches:
            if isinstance(item, Integral):
                idx = int(item)
            else:
                idx = geom_id_to_idx[id(item)]

            if bbox_intersects(feature_bboxes[idx], viewport_bbox):
                result.append(features[idx])

        return result

    print(f"STRtree build time: {build_ms:.2f} ms")
    print(f"{'Viewport':<22} {'Linear (ms)':>13} {'Grid (ms)':>11} {'STRtree (ms)':>13} {'Winner':>10} {'Features':>10}")
    print("-" * 88)

    for name, vp in viewports:
        t0 = time.perf_counter()
        for _ in range(RUNS):
            linear_result = linear_cull(features, vp)
        linear_ms = (time.perf_counter() - t0) / RUNS * 1000

        t0 = time.perf_counter()
        for _ in range(RUNS):
            grid_result = index.query(vp)
        grid_ms = (time.perf_counter() - t0) / RUNS * 1000

        t0 = time.perf_counter()
        for _ in range(RUNS):
            strtree_result = strtree_cull(vp)
        strtree_ms = (time.perf_counter() - t0) / RUNS * 1000

        if set(id(f) for f in linear_result) != set(id(f) for f in strtree_result):
            raise ValueError(f"STRtree result mismatch for {name}")

        timings = {"linear": linear_ms, "grid": grid_ms, "strtree": strtree_ms}
        winner = min(timings, key=timings.get)
        print(f"{name:<22} {linear_ms:>12.3f} {grid_ms:>10.3f} {strtree_ms:>12.3f} {winner:>10} {len(linear_result):>10,}")

    print("\nR-tree takeaway: STRtree usually wins on small or unevenly dense viewports because it adapts to feature distribution instead of using fixed 10-degree cells.")

except ImportError as exc:
    print("Shapely is not installed in this environment.")
    print("Install it, then rerun this cell:")
    print("  pip install shapely")
    print(f"Original import error: {exc}")

STRtree build time: 8.65 ms
Viewport                 Linear (ms)   Grid (ms)  STRtree (ms)     Winner   Features
----------------------------------------------------------------------------------------
World z2                     22.755     29.712       12.188    strtree     25,413
Europe z5                    21.689     13.973        4.599    strtree     12,626
France z7                    21.611      6.132        0.558    strtree      1,598
Paris z10                    21.057      1.428        0.032    strtree         51
Central Paris z13            20.256      0.786        0.018    strtree         11

R-tree takeaway: STRtree usually wins on small or unevenly dense viewports because it adapts to feature distribution instead of using fixed 10-degree cells.


## Check Your Understanding

Our `GridIndex` stores **references** to feature objects in multiple cells. It does not copy the features.

If a feature appears in 6 cells, how much extra memory does that use compared to storing it once? And why does deduplication in `query()` still need to happen even though we only stored references?

---

**Answer:** If a feature appears in 6 cells, the feature geometry/properties are still stored once. The extra index memory is 6 references to that same object, plus the list and tuple/set bookkeeping around those references; it is not 6 full copies of the GeoJSON feature.

Deduplication is still required because a query can touch several cells that all contain the same feature reference. Without the `seen` set, the result list could include the same feature multiple times, which would overcount it and draw it repeatedly on the map.

## Next

In [Module 05 — Zoom-Driven Layer Switching](../05-Zoom_Layer_Switching/README.md), we combine the LOD files with the grid index and switch both the data source and the index based on zoom level.